# Derive Security Table

This notebook pulls live IBKR portfolio positions and combines them with curated security-reference metadata plus fresh IBKR contract details.

Prerequisites:
- Run it with the `py313` kernel.
- Start TWS or IB Gateway locally.
- Enable API access and confirm the host, port, and client ID match your local settings.
- Each live notebook or script needs a unique `IB_CLIENT_ID`.
- If the optional single-contract lookup is ambiguous, tighten `PRIMARY_EXCHANGE` or set `CONID` directly.

In [30]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while not (project_root / "market_helper").exists():
    if project_root.parent == project_root:
        raise RuntimeError("Could not find project root containing the market_helper package.")
    project_root = project_root.parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

SYMBOL = "VIX"
SEC_TYPE = "IND"
PRIMARY_EXCHANGE = None
EXCHANGE = "CBOE"
CURRENCY = None
CONID = None
LOCAL_SYMBOL = None

IB_HOST = "127.0.0.1"
IB_PORT = 7497
IB_CLIENT_ID = 121
IB_ACCOUNT = ""

print(f"Project root: {project_root}")
print(
    "Optional single-contract lookup: "
    f"symbol={SYMBOL}, sec_type={SEC_TYPE}, exchange={EXCHANGE}, "
    f"primary_exchange={PRIMARY_EXCHANGE}, currency={CURRENCY}, conid={CONID}"
)
print(f"IBKR connection: host={IB_HOST}, port={IB_PORT}, client_id={IB_CLIENT_ID}, account={IB_ACCOUNT or '<default>'}")

Project root: /Users/kelvin/git_projects/market_helper
Optional single-contract lookup: symbol=VIX, sec_type=IND, exchange=CBOE, primary_exchange=None, currency=None, conid=None
IBKR connection: host=127.0.0.1, port=7497, client_id=121, account=<default>


In [31]:
from pprint import pprint

from IPython.display import display

try:
    import pandas as pd
except ModuleNotFoundError:
    pd = None

from market_helper.data_sources.ibkr.tws import TwsIbAsyncClient


lookup_client = TwsIbAsyncClient(
    host=IB_HOST,
    port=IB_PORT,
    client_id=IB_CLIENT_ID,
    account=IB_ACCOUNT,
)

try:
    lookup_client.connect()
    security_matches = lookup_client.search_securities(
        symbol=SYMBOL,
        sec_type=SEC_TYPE,
        exchange=EXCHANGE,
        primary_exchange=PRIMARY_EXCHANGE,
        currency=CURRENCY,
        conid=CONID,
        local_symbol=LOCAL_SYMBOL,
    )
finally:
    lookup_client.disconnect()

print(f"Returned {len(security_matches)} contract match(es).")

security_info = None

if not security_matches:
    print("No matches found. Tighten exchange/primary_exchange/local_symbol or set CONID.")
elif len(security_matches) == 1:
    security_info = security_matches[0]
    pprint(security_info)
elif pd is None:
    pprint(security_matches)
else:
    match_frame = pd.DataFrame(security_matches)
    preview_columns = [
        "conId",
        "symbol",
        "secType",
        "exchange",
        "primaryExchange",
        "currency",
        "localSymbol",
        "longName",
        "marketName",
        "validExchanges",
    ]
    ordered_columns = [column for column in preview_columns if column in match_frame.columns]
    remaining_columns = [column for column in match_frame.columns if column not in ordered_columns]
    display(match_frame.loc[:, ordered_columns + remaining_columns].reset_index(drop=True))
    print(
        "Multiple matches returned. Tighten PRIMARY_EXCHANGE/EXCHANGE/LOCAL_SYMBOL "
        "or set CONID to select one contract."
    )


Returned 1 contract match(es).
{'category': 'Volatility Index',
 'conId': 13455763,
 'currency': 'USD',
 'exchange': 'CBOE',
 'industry': 'Indices',
 'liquidHours': '20260404:CLOSED;20260405:CLOSED;20260406:0215-20260406:0815;20260406:0830-20260406:1600;20260407:0215-20260407:0815;20260407:0830-20260407:1600;20260408:0215-20260408:0815;20260408:0830-20260408:1600;20260409:0215-20260409:0815;20260409:0830-20260409:1600',
 'localSymbol': 'VIX',
 'longName': 'CBOE Volatility Index',
 'marketName': '',
 'minTick': 0.01,
 'orderTypes': 'ACTIVETIM,AD,ADJUST,ALERT,ALLOC,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,LMT,NONALGO,OCA,SCALE,SCALERST,WHATIF',
 'priceMagnifier': 1,
 'primaryExchange': '',
 'secType': 'IND',
 'subcategory': '*',
 'symbol': 'VIX',
 'tradingHours': '20260404:CLOSED;20260405:CLOSED;20260406:0215-20260406:0815;20260406:0830-20260406:1600;20260407:0215-20260407:0815;20260407:0830-20260407:1600;20260408:0215-20260408:0815;20260408:0830-2026

In [33]:
try:
    lookup_client.connect()
    security_matches = lookup_client.lookup_security(
        symbol=SYMBOL,
        sec_type=SEC_TYPE,
        exchange=EXCHANGE,
        primary_exchange=PRIMARY_EXCHANGE,
        currency=CURRENCY,
        conid=CONID,
        local_symbol=LOCAL_SYMBOL,
    )
finally:
    lookup_client.disconnect()

security_matches


{'conId': 13455763,
 'symbol': 'VIX',
 'secType': 'IND',
 'currency': 'USD',
 'exchange': 'CBOE',
 'primaryExchange': '',
 'localSymbol': 'VIX',
 'marketName': '',
 'minTick': 0.01,
 'priceMagnifier': 1,
 'orderTypes': 'ACTIVETIM,AD,ADJUST,ALERT,ALLOC,BASKET,BENCHPX,COND,CONDORDER,DAY,DEACT,DEACTDIS,DEACTEOD,GAT,GTC,GTD,GTT,HID,LMT,NONALGO,OCA,SCALE,SCALERST,WHATIF',
 'validExchanges': 'CBOE',
 'tradingHours': '20260404:CLOSED;20260405:CLOSED;20260406:0215-20260406:0815;20260406:0830-20260406:1600;20260407:0215-20260407:0815;20260407:0830-20260407:1600;20260408:0215-20260408:0815;20260408:0830-20260408:1600;20260409:0215-20260409:0815;20260409:0830-20260409:1600',
 'liquidHours': '20260404:CLOSED;20260405:CLOSED;20260406:0215-20260406:0815;20260406:0830-20260406:1600;20260407:0215-20260407:0815;20260407:0830-20260407:1600;20260408:0215-20260408:0815;20260408:0830-20260408:1600;20260409:0215-20260409:0815;20260409:0830-20260409:1600',
 'longName': 'CBOE Volatility Index',
 'industry': '